In [ ]:
from google.colab import drive
drive.mount('/content/drive')


DRIVE_ROOT = '/content/drive/My Drive/Siboubou/'
AUDIO_WORKING_DIR = os.path.join(DRIVE_ROOT, "siboubou_audios")

AUDIO_DIR = os.path.join(AUDIO_WORKING_DIR, "processed_audio")
MAPPING_CSV = os.path.join(AUDIO_WORKING_DIR, "processed_audio_mapper.csv")


AUDIO_COL = "audio_path"
TEXT_COL = "text"

import os
assert os.path.isdir(AUDIO_DIR), f"Audio folder not found: {AUDIO_DIR}"
assert os.path.isfile(MAPPING_CSV), f"Mapping CSV not found: {MAPPING_CSV}"


In [ ]:
!apt-get -qq update && apt-get -qq install -y ffmpeg

%cd /content
!git clone --depth 1 https://github.com/gokhaneraslan/chatterbox-finetuning.git
%cd /content/chatterbox-finetuning
!pip install -q -r requirements.txt
!pip install -q pandas soundfile librosa


In [ ]:
# Verifing audio files
import pandas as pd
import soundfile as sf
import os

df = pd.read_csv(MAPPING_CSV)
print(f"Number of rows in CSV: {len(df)}")

missing = []
for _, row in df.iterrows():
    wav_path = os.path.join(AUDIO_DIR, row[AUDIO_COL])
    if not os.path.isfile(wav_path):
        missing.append(wav_path)
        continue
    try:
        info = sf.info(wav_path)
    except Exception as e:
        missing.append(f"{wav_path} (unreadable: {e})")

if missing:
    print(f"There are {len(missing)} files")
else:
    print(f"No Files where missing.")


In [ ]:
# Convert audios to 16Khz mono
import shutil
import subprocess
import re

DATASET_DIR = "/content/chatterbox-finetuning/MyTTSDataset"
WAVS_DIR = os.path.join(DATASET_DIR, "wavs")
os.makedirs(WAVS_DIR, exist_ok=True)

def normalize_text(t: str) -> str:
    t = str(t).strip()
    t = re.sub(r"\s+", " ", t)
    return t

metadata_lines = []
skipped = 0

for i, row in df.iterrows():
    src_path = os.path.join(AUDIO_DIR, row[AUDIO_COL])
    if not os.path.isfile(src_path):
        skipped += 1
        continue

    raw_text = str(row[TEXT_COL])
    norm_text = normalize_text(raw_text)
    if not norm_text:
        skipped += 1
        continue

    file_id = f"dialect_{i:05d}"
    dst_path = os.path.join(WAVS_DIR, f"{file_id}.wav")

    subprocess.run(
        ["ffmpeg", "-y", "-loglevel", "error", "-i", src_path,
         "-ac", "1", "-ar", "16000", dst_path],
        check=True,
    )

    metadata_lines.append(f"{file_id}|{raw_text}|{norm_text}")

metadata_path = os.path.join(DATASET_DIR, "metadata.csv")
with open(metadata_path, "w", encoding="utf-8") as f:
    f.write("\n".join(metadata_lines) + "\n")

print(f"Wrote {len(metadata_lines)} entries ({skipped} skipped)")


In [ ]:
# configuring config.py

config_path = "/content/chatterbox-finetuning/src/config.py"
is_turbo = True
is_lora = True
ljspeech = True
json_format = False
preprocess = True

with open(config_path, "r", encoding="utf-8") as f:
    config_src = f.read()

# Dict to swap configuration booleans
replacements = {
    f"is_turbo: bool = {not is_turbo}": f"is_turbo: bool = {is_turbo}",
    f"is_lora: bool = {not is_lora}": f"is_lora: bool = {is_lora}",
    f"ljspeech = {not ljspeech}": f"ljspeech = {ljspeech}",
    f"json_format = {not json_format}": f"json_format = {json_format}",
    f"preprocess = {not preprocess}": f"preprocess = {preprocess}",
}

for old, new in replacements.items():
    config_src = config_src.replace(old, new)

with open(config_path, "w", encoding="utf-8") as f:
    f.write(config_src)

print("config.py was Updated")

In [ ]:
# update vocabulary size in config.py
import re

%cd /content/chatterbox-finetuning
result = !python setup.py

output = "\n".join(result)
match = re.search(r"new_vocab_size.*?(\d+)", output)

if match:
    vocab_size = match.group(1)
    config_src = open(config_path, encoding="utf-8").read()
    config_src = re.sub(
        r"new_vocab_size: int = \d+ if is_turbo else \d+",
        f"new_vocab_size: int = {vocab_size} if is_turbo else {vocab_size}",
        config_src,
    )
    open(config_path, "w", encoding="utf-8").write(config_src)
    print(f"new_vocab_size = {vocab_size}")
else:
    print("Cannot find new_vocab_size in setup.py output")

In [ ]:
# create speaker refrence
ref_src = os.path.join(WAVS_DIR, os.listdir(WAVS_DIR)[0])
os.makedirs("/content/chatterbox-finetuning/speaker_reference", exist_ok=True)
ref_dst = "/content/chatterbox-finetuning/speaker_reference/reference.wav"
shutil.copy(ref_src, ref_dst)


In [ ]:
# start fine tuning
%cd /content/chatterbox-finetuning
!python train.py


In [ ]:
#testing
import torch
import torchaudio
from src.chatterbox_.tts import ChatterboxTTS

device = "cuda" if torch.cuda.is_available() else "cpu"

model = ChatterboxTTS.from_local("/content/chatterbox-finetuning/checkpoints", device=device)

text = "سلام خويا, كيداير اليوم؟"

wav = model.generate(text, audio_prompt_path=ref_dst)
torchaudio.save("test_output.wav", wav, model.sr)

from IPython.display import Audio, display
display(Audio("test_output.wav"))